# bioseqkit demo

This notebook demonstrates the core features of **bioseqkit** on a real
sequence. It tries to download the *E. coli* K-12 MG1655 genome region (or
the human mitochondrion `chrM`) from NCBI; if there is no network access it
falls back to the bundled `example_data/sample.fa`.


## 1. Load data (NCBI with local fallback)

In [ ]:
from pathlib import Path
import bioseqkit as bsk

DATA = Path('example_data/sample.fa')
records = list(bsk.parse_fasta(str(DATA)))

# Optional: fetch a real genome from NCBI (needs internet).
try:
    from bioseqkit.entrez import efetch_fasta
    import io
    fasta_text = efetch_fasta('NC_012920.1', email='you@example.com')  # human chrM
    records = list(bsk.parse_fasta(io.StringIO(fasta_text)))
    print('Downloaded from NCBI:', records[0].id, records[0].description)
except Exception as exc:
    print('Using local example data (no network):', exc)

for r in records:
    print(r.id, len(r), r.description)


## 2. Basic statistics

In [ ]:
stats = bsk.sequence_stats(r.sequence for r in records)
import json
print(json.dumps(stats.as_dict(), indent=2))


## 3. GC content & length distribution

In [ ]:
import matplotlib.pyplot as plt

gc = [bsk.gc_content(r.sequence) for r in records]
lengths = [len(r) for r in records]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar([r.id for r in records], gc, color='steelblue')
axes[0].set_ylabel('GC content'); axes[0].set_title('GC content per sequence')
axes[1].bar([r.id for r in records], lengths, color='indianred')
axes[1].set_ylabel('length (bp)'); axes[1].set_title('Sequence length')
plt.tight_layout(); plt.show()


## 4. k-mer spectrum

In [ ]:
from collections import Counter
k = 4
counts = Counter()
for r in records:
    counts += bsk.count_kmers(r.sequence, k, canonical=True)

print('Top-10 canonical %d-mers:' % k)
for kmer, c in bsk.top_kmers(counts, 10):
    print(f'  {kmer}\t{c}')

# k-mer spectrum: histogram of k-mer multiplicities
spectrum = Counter(counts.values())
xs = sorted(spectrum)
plt.figure(figsize=(6, 4))
plt.bar(xs, [spectrum[x] for x in xs], color='seagreen')
plt.xlabel('k-mer multiplicity'); plt.ylabel('# distinct k-mers')
plt.title(f'{k}-mer spectrum'); plt.tight_layout(); plt.show()


## 5. Six-frame translation vs. known CDS

In [ ]:
seq = records[0].sequence
for frame in bsk.six_frame_translation(seq):
    print(frame.name, frame.protein[:60])


## 6. Minimizer distribution

In [ ]:
mins = bsk.minimizers(records[0].sequence, k=8, w=5)
positions = [p for p, _ in mins]
print(f'{len(mins)} minimizers selected from {len(records[0])} bp')
plt.figure(figsize=(8, 2))
plt.eventplot(positions, colors='black')
plt.xlabel('position (bp)'); plt.title('Minimizer positions'); plt.yticks([])
plt.tight_layout(); plt.show()


## 7. Random access with the FAI-like index

In [ ]:
from bioseqkit.index import build_faidx
idx = build_faidx(str(DATA))
name = idx.names()[0]
print('Index records:', idx.names())
print('Fetch', name, '1-20:', idx.fetch(name, 1, 20))


## 8. Multithreaded k-mer benchmark
Compare single-process vs. multi-process k-mer counting on a larger sequence.

In [ ]:
import time
big = (records[0].sequence * 2000)
print(f'sequence length: {len(big):,} bp')

t0 = time.perf_counter(); c1 = bsk.count_kmers(big, 8, canonical=True); t1 = time.perf_counter()
c4 = bsk.count_kmers_parallel([big], 8, canonical=True, workers=4); t2 = time.perf_counter()
print(f'serial:   {t1 - t0:.3f}s')
print(f'parallel: {t2 - t1:.3f}s  (identical result: {c1 == c4})')
